In [43]:
# NLP Text Preprocessing Script

In [44]:
import pandas as pd 
import numpy as np
import os
import re
import string

In [53]:
# Step 1 : Load the dataset

In [46]:
# Data path
DATA_PATH = r"C:\Users\HIMANSHU VYAS\Desktop\consumer-complaint-intelligence\data\processed"

In [47]:
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

EXPECTED_COLUMNS = {"has_narrative", "Consumer complaint narrative"}  # add others you rely on downstream

def load_chunk(file_path: str | Path) -> pd.DataFrame:
    """Load a single complaint chunk with validation and clear failure messages."""
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Chunk not found: {file_path}")

    if file_path.stat().st_size == 0:
        raise ValueError(f"Chunk is empty (0 bytes): {file_path}")

    try:
        df = pd.read_csv(file_path, encoding="utf-8")
    except UnicodeDecodeError:
        logger.warning(f"UTF-8 decode failed for {file_path.name}, retrying with latin-1")
        df = pd.read_csv(file_path, encoding="latin-1")
    except pd.errors.EmptyDataError:
        raise ValueError(f"Chunk has no columns/data: {file_path}")
    except pd.errors.ParserError as e:
        raise ValueError(f"Malformed CSV in {file_path.name}: {e}")

    if df.empty:
        logger.warning(f"{file_path.name} loaded but has 0 rows")

    missing_cols = EXPECTED_COLUMNS - set(df.columns)
    if missing_cols:
        raise ValueError(f"{file_path.name} is missing expected columns: {missing_cols}")

    logger.info(f"Loaded {file_path.name}: {df.shape[0]} rows, {df.shape[1]} cols")
    return df


file_path = os.path.join(DATA_PATH, "complaints_part_0001.csv")
df = load_chunk(file_path)

C:\Users\HIMANSHU VYAS\AppData\Local\Temp\ipykernel_19752\2695549607.py:20: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding="utf-8")
2026-09-02 13:55:43,941 | INFO | Loaded complaints_part_0001.csv: 200000 rows, 16 cols


In [48]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,has_narrative,has_public_response
0,2024-07-22,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CT,Web,2024-07-22,Closed with non-monetary relief,Yes,9584621,False,True
1,2024-07-22,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,NaN,"EQUIFAX, INC.",CT,Web,2024-07-22,Closed with explanation,Yes,9580257,False,False
2,2024-07-22,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,CT,Web,2024-07-22,Closed with non-monetary relief,Yes,9584634,False,True
3,2024-09-03,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,NaN,"EQUIFAX, INC.",CT,Web,2024-09-03,Closed with explanation,Yes,9999666,False,False
4,2024-09-03,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CT,Web,2024-09-03,Closed with non-monetary relief,Yes,9995854,False,True


In [49]:
df.shape

(200000, 16)

In [50]:

logger = logging.getLogger(__name__)

def narrative_coverage_report(df: pd.DataFrame, column: str = "has_narrative") -> pd.DataFrame:
    """
    Summarize how many rows have a narrative vs not, as counts and percentages.

    Returns a small DataFrame with columns: [count, percentage]
    indexed by the values of `column` (typically True/False, but also
    handles NaN and unexpected values safely).
    """
    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found. Available columns: {df.columns.tolist()}")

    if df.empty:
        logger.warning("DataFrame is empty — returning empty coverage report")
        return pd.DataFrame(columns=["count", "percentage"])

    counts = df[column].value_counts(dropna=False)
    percentages = df[column].value_counts(normalize=True, dropna=False) * 100

    report = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    })

    # Flag anything that isn't a clean True/False — signals upstream data issues
    unexpected_values = [v for v in report.index if v not in (True, False)]
    if unexpected_values:
        logger.warning(f"Unexpected values in '{column}': {unexpected_values} — check upstream labeling")

    return report


report = narrative_coverage_report(df)
print(report)

                count  percentage
has_narrative                    
False          196230       98.12
True             3770        1.88


In [52]:

logger = logging.getLogger(__name__)

def extract_narratives(
    df: pd.DataFrame,
    flag_column: str = "has_narrative",
    text_column: str = "Consumer complaint narrative",
) -> pd.Series:
    """
    Extract non-null narrative text for rows flagged as having a narrative.

    Returns a Series of narrative strings, indexed by original row index
    (index is preserved so you can map cleaned text back to df later).
    """
    missing_cols = {flag_column, text_column} - set(df.columns)
    if missing_cols:
        raise KeyError(f"Missing expected column(s): {missing_cols}. Available: {df.columns.tolist()}")

    if df.empty:
        logger.warning("DataFrame is empty — returning empty narrative series")
        return pd.Series(dtype="object")

    # Rows flagged True but where the text is actually missing = a labeling bug upstream
    flagged_true = df[flag_column] == True
    flagged_but_missing = flagged_true & df[text_column].isna()
    if flagged_but_missing.sum() > 0:
        logger.warning(
            f"{flagged_but_missing.sum()} row(s) marked has_narrative=True "
            f"but '{text_column}' is null — check upstream flagging logic"
        )

    narratives = df.loc[flagged_true, text_column].dropna()

    # Catch rows that are technically non-null but empty/whitespace-only strings
    blank_mask = narratives.str.strip().eq("")
    if blank_mask.sum() > 0:
        logger.warning(f"{blank_mask.sum()} narrative(s) are blank/whitespace-only — excluding them")
        narratives = narratives[~blank_mask]

    logger.info(f"Extracted {len(narratives)} usable narratives out of {len(df)} rows")

    if len(narratives) == 0:
        logger.warning("No usable narratives found in this chunk")

    return narratives


narratives = extract_narratives(df)
print("Number of narratives:", len(narratives))

2026-09-02 14:01:15,948 | INFO | Extracted 3770 usable narratives out of 200000 rows


Number of narratives: 3770


In [54]:
# Step 2: Preprocess the text data

In [55]:

logger = logging.getLogger(__name__)

def get_sample_narrative(narratives: pd.Series, index: int = 0, preview_chars: int = 500) -> str:
    """
    Safely retrieve a single narrative by position for inspection/testing.

    Args:
        narratives: Series of narrative text (e.g. output of extract_narratives()).
        index: positional index into the series (0 = first).
        preview_chars: how much of the text to show in the log preview.

    Returns:
        The full narrative text as a string.
    """
    if narratives.empty:
        raise ValueError("narratives is empty — nothing to sample. Check extract_narratives() output.")

    if index < 0 or index >= len(narratives):
        raise IndexError(
            f"index={index} out of range for narratives of length {len(narratives)}"
        )

    text = narratives.iloc[index]

    if not isinstance(text, str):
        raise TypeError(f"Expected str at index {index}, got {type(text)}: {text!r}")

    if not text.strip():
        logger.warning(f"Narrative at index {index} is blank/whitespace-only")

    logger.info(
        f"Sampled narrative {index} (length={len(text)} chars): "
        f"{text[:preview_chars]}{'...' if len(text) > preview_chars else ''}"
    )

    return text


text = get_sample_narrative(narratives, index=0)
print(text)

2026-09-02 14:03:16,256 | INFO | Sampled narrative 0 (length=3796 chars): XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized access and account tampering involving my personal information. Over the past one to two years, I experienced a pattern of unexpected missed payments across several of my credit accounts. During this period, I was in a relationship where the individual had access to my personal laptop and login credentia...


XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized access and account tampering involving my personal information. Over the past one to two years, I experienced a pattern of unexpected missed payments across several of my credit accounts. During this period, I was in a relationship where the individual had access to my personal laptop and login credentials. After the relationship ended, I discovered that this person had deliberately tampered with my accounts disabling autopay, removing saved payment methods, and altering my email and notification settings without my knowledge or consent. The individual later admitted to me directly over the phone that they accessed my accounts maliciously and interfered with my financial information. These actions were taken without my authorization and directly caused the missed payments on accounts that were 

In [56]:
# Setep 2.1: Lowercase the text

In [57]:

logger = logging.getLogger(__name__)

def lowercase_text(text: str) -> str:
    """
    Lowercase narrative text safely.

    Returns an empty string for missing/non-string input rather than
    raising, since this is meant to run inside a larger cleaning pipeline
    where one bad row shouldn't stop the whole chunk.
    """
    if not isinstance(text, str):
        logger.warning(f"Expected str, got {type(text)} — returning empty string")
        return ""

    if not text.strip():
        return text  # nothing to lowercase, but don't warn — blank is valid, already flagged upstream

    return text.lower()


text_lower = lowercase_text(text)

print("Original:")
print(text[:300])
print("\nLowercase:")
print(text_lower[:300])

Original:
XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized access and account tampering involving my personal information. Over the past one to two years, I experie

Lowercase:
xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx i am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized access and account tampering involving my personal information. over the past one to two years, i experie


In [58]:
# Step 2.3: Correction handling 
# A contraction is a shortened form of two words

In [60]:
contraction_map = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "wouldn't": "would not",
    "shouldn't": "should not",
    "couldn't": "could not",
    "mustn't": "must not"
}

In [61]:
def expand_contractions(text):
    for contraction, replacement in contraction_map.items():
        text = text.replace(contraction, replacement)

    return text

In [62]:
text_expanded = expand_contractions(text_lower)

print("Before:")
print(text_lower)

print("\nAfter:")
print(text_expanded)

Before:
xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx i am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized access and account tampering involving my personal information. over the past one to two years, i experienced a pattern of unexpected missed payments across several of my credit accounts. during this period, i was in a relationship where the individual had access to my personal laptop and login credentials. after the relationship ended, i discovered that this person had deliberately tampered with my accounts disabling autopay, removing saved payment methods, and altering my email and notification settings without my knowledge or consent. the individual later admitted to me directly over the phone that they accessed my accounts maliciously and interfered with my financial information. these actions were taken without my authorization and directly caused the missed payments on accounts th

In [63]:
found_contractions = [
    contraction
    for contraction in contraction_map
    if contraction in text_lower
]

print(found_contractions)

[]


In [64]:
# Step 2.4: Tokenization

In [65]:

logger = logging.getLogger(__name__)

# Matches sequences of letters/digits as whole tokens — this is what actually
# does your "punctuation removal" for you (see note below).
_TOKEN_PATTERN = re.compile(r"\b[a-zA-Z0-9]+\b")

def tokenize_text(text: str, min_token_length: int = 1) -> list[str]:
    """
    Split cleaned text into tokens (words/numbers), dropping punctuation.

    Args:
        text: text that has already been lowercased and had contractions expanded.
        min_token_length: drop tokens shorter than this (e.g. set to 2 to drop
            stray single letters left over from OCR/typos — default keeps everything).

    Returns:
        List of token strings. Empty list if input is missing/blank.
    """
    if not isinstance(text, str):
        logger.warning(f"Expected str, got {type(text)} — returning empty token list")
        return []

    if not text.strip():
        return []

    tokens = _TOKEN_PATTERN.findall(text)

    if min_token_length > 1:
        tokens = [t for t in tokens if len(t) >= min_token_length]

    if not tokens:
        logger.warning("Text produced zero tokens after tokenization — check input")

    return tokens


tokens = tokenize_text(text_expanded)
print(f"Token count: {len(tokens)}")
print(tokens[:30])

Token count: 585
['xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'i', 'am', 'writing', 'to', 'formally', 'dispute', 'multiple', 'late', 'payments', 'reported', 'on', 'my', 'credit', 'file', 'which', 'are']


In [66]:

logger = logging.getLogger(__name__)

def remove_stopwords(tokens: list[str], stopword_set: set[str]) -> list[str]:
    """
    Remove stopwords from a token list.

    Args:
        tokens: list of tokens (output of tokenize_text()).
        stopword_set: the stopword set to filter against — pass CUSTOM_STOPWORDS,
            not the raw NLTK set, so negation words are preserved.

    Returns:
        Filtered token list. Empty list stays empty (not an error).
    """
    if not tokens:
        return []

    if not isinstance(stopword_set, set):
        logger.warning("stopword_set is not a set — converting for O(1) lookups")
        stopword_set = set(stopword_set)

    filtered = [word for word in tokens if word not in stopword_set]

    if not filtered and tokens:
        logger.warning("All tokens were removed as stopwords — check input narrative")

    return filtered


tokens_without_stopwords = remove_stopwords(tokens, custom_stop_words)

print("Tokens before:", len(tokens))
print("Tokens after:", len(tokens_without_stopwords))
print("\nRemaining tokens:")
print(tokens_without_stopwords[:100])

Tokens before: 585
Tokens after: 367

Remaining tokens:
['xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'xxxx', 'writing', 'formally', 'dispute', 'multiple', 'late', 'payments', 'reported', 'credit', 'file', 'inaccurate', 'due', 'unauthorized', 'access', 'account', 'tampering', 'involving', 'personal', 'information', 'over', 'past', 'one', 'two', 'years', 'experienced', 'pattern', 'unexpected', 'missed', 'payments', 'across', 'several', 'credit', 'accounts', 'during', 'period', 'relationship', 'individual', 'access', 'personal', 'laptop', 'login', 'credentials', 'after', 'relationship', 'ended', 'discovered', 'person', 'deliberately', 'tampered', 'accounts', 'disabling', 'autopay', 'removing', 'saved', 'payment', 'methods', 'altering', 'email', 'notification', 'settings', 'without', 'knowledge', 'consent', 'individual', 'later', 'admitted', 'directly', 'over', 'phone', 'accessed', 'accounts', 'maliciously', 'interfered', 'financi

In [68]:
from collections import Counter

def stopword_removal_diff(tokens: list[str], filtered_tokens: list[str]) -> dict:
    """
    Compare before/after token lists and report exactly what was removed.
    """
    removed = [t for t in tokens if t not in filtered_tokens]
    removed_counts = Counter(removed)

    return {
        "total_before": len(tokens),
        "total_after": len(filtered_tokens),
        "total_removed": len(removed),
        "pct_removed": round(len(removed) / len(tokens) * 100, 1) if tokens else 0,
        "unique_words_removed": len(removed_counts),
        "most_common_removed": removed_counts.most_common(15),
    }


diff = stopword_removal_diff(tokens, tokens_without_stopwords)

print(f"\nRemoved {diff['total_removed']} of {diff['total_before']} tokens "
      f"({diff['pct_removed']}%)")
print(f"Unique stopwords removed: {diff['unique_words_removed']}")
print("\nMost frequently removed words:")
for word, count in diff["most_common_removed"]:
    print(f"  {word:15} removed {count} time(s)")


Removed 218 of 585 tokens (37.3%)
Unique stopwords removed: 47

Most frequently removed words:
  the             removed 27 time(s)
  to              removed 20 time(s)
  my              removed 17 time(s)
  of              removed 17 time(s)
  and             removed 16 time(s)
  i               removed 13 time(s)
  this            removed 9 time(s)
  or              removed 9 time(s)
  these           removed 7 time(s)
  that            removed 6 time(s)
  in              removed 5 time(s)
  s               removed 5 time(s)
  for             removed 5 time(s)
  am              removed 4 time(s)
  a               removed 4 time(s)


In [69]:
negation_check = {w: (w in tokens_without_stopwords) for w in ["not", "no", "never"] if w in tokens}
print("\nNegation words present in original tokens and did they survive?")
print(negation_check)


Negation words present in original tokens and did they survive?
{'not': True, 'no': True}


In [70]:
def check_negation_survival(narratives_sample, n=20):
    """Run the full pipeline (minus stopword-diff) across n narratives and
    check negation-word survival rate."""
    results = []
    for i in range(min(n, len(narratives_sample))):
        raw = narratives_sample.iloc[i]
        text = expand_contractions(lowercase_text(raw))
        toks = tokenize_text(text)
        filtered = remove_stopwords(toks, custom_stop_words)

        for neg_word in ["not", "no", "never", "cannot"]:
            if neg_word in toks:
                results.append({
                    "narrative_idx": i,
                    "word": neg_word,
                    "survived": neg_word in filtered
                })
    return pd.DataFrame(results)


check_df = check_negation_survival(narratives, n=20)
print(check_df)
print("\nAny failures?", (check_df["survived"] == False).any())
print(check_df.groupby("word")["survived"].mean() * 100, "% survival rate")

    narrative_idx    word  survived
0               0     not      True
1               0      no      True
2               1     not      True
3               1      no      True
4               2     not      True
5               2      no      True
6               3     not      True
7               3      no      True
8               4     not      True
9               5     not      True
10              5      no      True
11              5   never      True
12              6     not      True
13              7     not      True
14              7      no      True
15              8     not      True
16              9      no      True
17             10     not      True
18             10   never      True
19             11     not      True
20             12     not      True
21             13   never      True
22             14  cannot      True
23             15   never      True
24             17     not      True
25             19     not      True
26             19   never   

In [71]:
# for xxxx this pattern

In [75]:
def clean_narrative(text: str) -> str:
    text = lowercase_text(text)
    text = expand_contractions(text)
    tokens = tokenize_text(text)
    tokens = remove_redaction_tokens(tokens)
    tokens = remove_stopwords(tokens, custom_stop_words)
    return " ".join(tokens)

In [76]:
x_pattern_counts = Counter()
for narrative in narratives:
    found = re.findall(r"\bx{2,}\b", narrative.lower())
    x_pattern_counts.update(found)

print(x_pattern_counts.most_common(10))

[('xxxx', 43088), ('xx', 13502), ('xxxxxxxx', 856), ('xxxxxx', 28), ('xxxxxxxxxxxx', 6), ('xxxxx', 2)]


In [73]:
def remove_redaction_tokens(tokens: list[str]) -> list[str]:
    """Remove CFPB redaction placeholders (any run of 2+ x's) — no semantic content."""
    return [t for t in tokens if not re.fullmatch(r"x{2,}", t)]

In [77]:
sample_clean = clean_narrative(narratives.iloc[0])  # uses the updated pipeline
print(sample_clean[:300])

writing formally dispute multiple late payments reported credit file inaccurate due unauthorized access account tampering involving personal information over past one two years experienced pattern unexpected missed payments across several credit accounts during period relationship individual access 


In [81]:
df["clean_narrative"] = ""
df.loc[narratives.index, "clean_narrative"] = narratives.apply(clean_narrative)

empty_after_cleaning = (df.loc[narratives.index, "clean_narrative"] == "").sum()
print(f"Empty after cleaning: {empty_after_cleaning} / {len(narratives)}")

for i in range(6):
    print(f"\n--- Narrative {i} ---")
    print("RAW:", narratives.iloc[i][:200])
    print("CLEAN:", df.loc[narratives.index, "clean_narrative"].iloc[i][:200])

Empty after cleaning: 0 / 3770

--- Narrative 0 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized acces
CLEAN: writing formally dispute multiple late payments reported credit file inaccurate due unauthorized access account tampering involving personal information over past one two years experienced pattern une

--- Narrative 1 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized acces
CLEAN: writing formally dispute multiple late payments reported credit file inaccurate due unauthorized access account tampering involving personal information over past one two years experienced pattern une

--- Narrative 2 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I a

In [82]:
# Lemmatization

In [83]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

True

In [84]:
from nltk.corpus import wordnet

def get_wordnet_pos(nltk_tag: str) -> str:
    """Map NLTK's POS tag to what WordNetLemmatizer expects.
    Without this, WordNetLemmatizer assumes everything is a noun, so verbs
    like 'disputed' or 'reported' won't reduce to their root form."""
    if nltk_tag.startswith("J"):
        return wordnet.ADJ
    elif nltk_tag.startswith("V"):
        return wordnet.VERB
    elif nltk_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [85]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

_lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens: list[str]) -> list[str]:
    """Reduce tokens to their dictionary root form, POS-aware."""
    if not tokens:
        return []
    tagged = pos_tag(tokens)
    return [_lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

In [86]:
sample_tokens = ["payments", "reported", "disputed", "accounts", "was", "removing", "issues"]
print(lemmatize_tokens(sample_tokens))

['payment', 'report', 'disputed', 'account', 'be', 'remove', 'issue']


In [87]:
def clean_narrative(text: str) -> str:
    text = lowercase_text(text)
    text = expand_contractions(text)
    tokens = tokenize_text(text)
    tokens = remove_redaction_tokens(tokens)
    tokens = remove_stopwords(tokens, custom_stop_words)
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

In [88]:
sample_clean = clean_narrative(narratives.iloc[0])
print(sample_clean[:300])

write formally dispute multiple late payment report credit file inaccurate due unauthorized access account tamper involve personal information over past one two year experience pattern unexpected miss payment across several credit account during period relationship individual access personal laptop 


In [89]:
df["clean_narrative"] = ""
df.loc[narratives.index, "clean_narrative"] = narratives.apply(clean_narrative)

empty_after_cleaning = (df.loc[narratives.index, "clean_narrative"] == "").sum()
print(f"Empty after cleaning: {empty_after_cleaning} / {len(narratives)}")

for i in range(6):
    print(f"\n--- Narrative {i} ---")
    print("RAW:", narratives.iloc[i][:200])
    print("CLEAN:", df.loc[narratives.index, "clean_narrative"].iloc[i][:200])

Empty after cleaning: 0 / 3770

--- Narrative 0 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized acces
CLEAN: write formally dispute multiple late payment report credit file inaccurate due unauthorized access account tamper involve personal information over past one two year experience pattern unexpected miss

--- Narrative 1 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I am writing to formally dispute multiple late payments reported on my credit file, which are inaccurate due to unauthorized acces
CLEAN: write formally dispute multiple late payment report credit file inaccurate due unauthorized access account tamper involve personal information over past one two year experience pattern unexpected miss

--- Narrative 2 ---
RAW: XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX I a

In [90]:
# Validation

In [91]:
# Token reduction validation

for i in range(6):
    raw_tokens = tokenize_text(
        expand_contractions(
            lowercase_text(narratives.iloc[i])
        )
    )

    clean_tokens = df.loc[
        narratives.index, "clean_narrative"
    ].iloc[i].split()

    removed = len(raw_tokens) - len(clean_tokens)
    reduction_pct = (removed / len(raw_tokens)) * 100

    print(f"\n--- Narrative {i} ---")
    print("Raw tokens:", len(raw_tokens))
    print("Clean tokens:", len(clean_tokens))
    print("Tokens removed:", removed)
    print(f"Reduction: {reduction_pct:.2f}%")


--- Narrative 0 ---
Raw tokens: 585
Clean tokens: 353
Tokens removed: 232
Reduction: 39.66%

--- Narrative 1 ---
Raw tokens: 585
Clean tokens: 353
Tokens removed: 232
Reduction: 39.66%

--- Narrative 2 ---
Raw tokens: 585
Clean tokens: 353
Tokens removed: 232
Reduction: 39.66%

--- Narrative 3 ---
Raw tokens: 310
Clean tokens: 139
Tokens removed: 171
Reduction: 55.16%

--- Narrative 4 ---
Raw tokens: 343
Clean tokens: 199
Tokens removed: 144
Reduction: 41.98%

--- Narrative 5 ---
Raw tokens: 263
Clean tokens: 137
Tokens removed: 126
Reduction: 47.91%


In [92]:
# Verify xxxx reduction

In [93]:
# Check whether redaction tokens remain

redaction_pattern = r"^x{2,}$"

remaining_redactions = []

for idx in narratives.index:
    clean_text = df.loc[idx, "clean_narrative"]

    tokens = clean_text.split()

    for token in tokens:
        if re.fullmatch(redaction_pattern, token):
            remaining_redactions.append(token)

print("Remaining redaction tokens:", len(remaining_redactions))

if remaining_redactions:
    print("Examples:", remaining_redactions[:20])
else:
    print("✓ No redaction tokens remain.")

Remaining redaction tokens: 0
✓ No redaction tokens remain.


In [94]:
# negative words

In [95]:
# Check negation preservation

negation_words = {
    "not",
    "no",
    "never",
    "neither",
    "nor",
    "without"
}

for i in range(6):
    
    clean_tokens = df.loc[
        narratives.index, "clean_narrative"
    ].iloc[i].split()

    found_negations = [
        word for word in clean_tokens
        if word in negation_words
    ]

    print(f"\n--- Narrative {i} ---")
    print("Negation words found:", found_negations)


--- Narrative 0 ---
Negation words found: ['without', 'without', 'no', 'not', 'not', 'nor', 'not']

--- Narrative 1 ---
Negation words found: ['without', 'without', 'no', 'not', 'not', 'nor', 'not']

--- Narrative 2 ---
Negation words found: ['without', 'without', 'no', 'not', 'not', 'nor', 'not']

--- Narrative 3 ---
Negation words found: ['no', 'no', 'not']

--- Narrative 4 ---
Negation words found: ['not']

--- Narrative 5 ---
Negation words found: ['not', 'no', 'not', 'not', 'never', 'never', 'never', 'not']


In [96]:
# Domain terms

In [97]:
# Check important domain terms

domain_terms = {
    "credit",
    "payment",
    "account",
    "report",
    "debt",
    "loan",
    "bank",
    "consumer",
    "company",
    "cfpb",
    "ftc"
}

for i in range(6):

    clean_tokens = df.loc[
        narratives.index, "clean_narrative"
    ].iloc[i].split()

    found_terms = [
        word for word in clean_tokens
        if word in domain_terms
    ]

    print(f"\n--- Narrative {i} ---")
    print("Domain terms found:", found_terms)


--- Narrative 0 ---
Domain terms found: ['payment', 'report', 'credit', 'account', 'payment', 'credit', 'account', 'account', 'payment', 'account', 'payment', 'account', 'account', 'account', 'account', 'report', 'credit', 'payment', 'credit', 'report', 'report', 'consumer', 'ftc', 'report', 'consumer', 'account', 'payment', 'payment', 'report', 'account', 'report', 'payment', 'account', 'report', 'ftc', 'consumer', 'cfpb', 'ftc']

--- Narrative 1 ---
Domain terms found: ['payment', 'report', 'credit', 'account', 'payment', 'credit', 'account', 'account', 'payment', 'account', 'payment', 'account', 'account', 'account', 'account', 'report', 'credit', 'payment', 'credit', 'report', 'report', 'consumer', 'ftc', 'report', 'consumer', 'account', 'payment', 'payment', 'report', 'account', 'report', 'payment', 'account', 'report', 'ftc', 'consumer', 'cfpb', 'ftc']

--- Narrative 2 ---
Domain terms found: ['payment', 'report', 'credit', 'account', 'payment', 'credit', 'account', 'account', '

In [98]:
# Lemmatization Validation

In [99]:
# Check lemmatization

lemma_test_words = [
    "payments",
    "reported",
    "accounts",
    "issues",
    "complaints",
    "disputes",
    "involving",
    "removed",
    "companies",
    "transactions"
]

print("Lemmatization results:\n")

for word in lemma_test_words:
    result = lemmatize_tokens([word])
    print(f"{word:15} → {result[0]}")

Lemmatization results:

payments        → payment
reported        → report
accounts        → account
issues          → issue
complaints      → complaint
disputes        → dispute
involving       → involve
removed         → remove
companies       → company
transactions    → transaction
